# DSML 4220 - Lab 10: A simple Agent with Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

In this lab we will use Ollama to create a simple agent armed with tools in order to help carry out tasks on our behalf. This notebook is based on the short blog posts/tutorials found [here](https://www.cohorte.co/blog/using-ollama-with-python-step-by-step-guide) and [here](https://towardsdatascience.com/ai-agents-from-zero-to-hero-part-1/).


### Lab 10 Assignment/Task
There are a few questions below that require some additional code to be written so that your agent can carry out other operations besides just addition.

Let's start out by setting up Ollama to run in Colab. If you run this notebook locally and already have Ollama running, then you can skip these steps.

In [55]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,201 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
106 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire o

The following two modules we'll need later on, but we install them here because Colab may ask to restart after they are installed with `pip`. It's better to restart at the beginning than to restart half-way through.

In [56]:
!pip install langchain_community
!pip install -U duckduckgo-search
!pip install -U ddgs

Now we need to get the Ollama server running. Run the following code block to do this.

In [57]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Next, let's pull the model we want to use, Llama 3.2 with 1 billion parameters.

In [58]:
!ollama pull llama3.2:1b

Then, install the Ollama Python api.

In [59]:
!pip install ollama

Finally, get started with using Ollama from Python.

In [60]:
import ollama

Now, let's define a __tool__ for the agent/model to use.

In [68]:
# Tool function to add two numbers
def add_two_numbers(a: int, b: int) -> int:
    a_int = int(a)
    b_int = int(b)
    return a_int + b_int

Next, let's set up the system prompt and an initial user prompt/question for the agent/model.

In [69]:
# System prompt to inform the model about the tool is usage
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."
}

# A sample of user input asking a math question
user_message = {
    "role": "user",
    "content": "What is 90999999 + 10000001?"
}

messages = [system_message, user_message]
messages

[{'role': 'system',
  'content': "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."},
 {'role': 'user', 'content': 'What is 90999999 + 10000001?'}]

Ask the agent/model to respond.

In [70]:
# Ask llama3.2 to respond
response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers]
)

In [71]:
response.message

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='add_two_numbers', arguments={'a': '90999999', 'b': '10000001'}))])

In [72]:
response.message.content

''

In [73]:
# Check if the model called a function
if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        func_name = tool_call.function.name   # e.g., "add_two_numbers"
        args = tool_call.function.arguments   # e.g., {"a": 10, "b": 10}
        # If the function name matches and we have it in our tools, execute it:
        if func_name == "add_two_numbers":
            result = add_two_numbers(**args)
            print("Function output:", result)




Function output: 101000000


---

### Q1: Does the above output look correct? Does it look like the sum of the numbers 90999999 and 10000001? Why is it not correct?

(Hint: there is nothing wrong with the model/agent here, but rather the tool implementation; namely, Python's [type hints](https://docs.python.org/3/library/typing.html) are not a guarantee that the correct/intended data type is used, so you may need to add some type casting inside of the function `add_two_numbers`)

Answer: It does not appear to be correct, as the numbers are concatinated and not a sum of 90999999 and 10000001. The type hint seems to not be using the int data type for a and b, but instead using strings for both, which would acount for the concatenation.

---

In [74]:
# Complete the agent's tool call and allow the model to use output to formulate an answer
""" (Continuing from previous code) """
available_functions = {"add_two_numbers": add_two_numbers, "multiply_two_numbers": multiply_two_numbers}

""" System prompt to inform the model about the tool is usage """

""" Model's initial response after possibly invoking the tool """
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

""" If a tool was called, handle it """
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): I found that to add 90999999 and 10000001, the result is 101000000.


---

### Q2: Try running the code cell below. Does it return the expect result? If note, then add/modify the necessary code to allow Llama3.2 to use its  multiplication tool. Then rerun your code cell below; now did it output the expected result?

Answer: It did not output the expected result, the multiply_two_numbers function just did a pass so it didn't know how to do the multiplication. When corrected, it still did not run as expected - the math logic is incorrect.

---

In [79]:
# Implement a multiplication function by replacing the `pass` statement below with the correct return statement
def multiply_two_numbers(a: int, b: int) -> int:
    a_int = int(a)
    b_int = int(b)
    return a_int * b_int

""" System prompt to inform the model about the tool is usage """
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do addition by calling the function 'add_two_numbers' or multiplication by calling the function 'multiply_two_numbers'."
}
# User asks a question that involves a calculation
user_message = {
    "role": "user",
    "content": "What is 10001 times 6?"
}

messages = [system_message, user_message]

response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers, multiply_two_numbers]  # pass the actual function object as a tool
)

# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {"add_two_numbers": add_two_numbers, "multiply_two_numbers": multiply_two_numbers}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): To calculate "10001 times 6", I added two numbers together.

Here's a step-by-step calculation:

1. Multiply 10,000 by 6: 
   10,000 x 6 = 60,000
2. Add 1 to the result:
   60,000 + 1 = 60,001

The final answer is 60,001.


In [80]:
follow_up.message

Message(role='assistant', content='To calculate "10001 times 6", I added two numbers together.\n\nHere\'s a step-by-step calculation:\n\n1. Multiply 10,000 by 6: \n   10,000 x 6 = 60,000\n2. Add 1 to the result:\n   60,000 + 1 = 60,001\n\nThe final answer is 60,001.', thinking=None, images=None, tool_name=None, tool_calls=None)

Next let's equip our agent to retrieve external information, which will require a few more tools to be able to search the web.

In [81]:
from langchain_community.tools import DuckDuckGoSearchResults


def search_web(query: str) -> str:
  return DuckDuckGoSearchResults(backend="news").run(query)

tool_search_web = {'type':'function', 'function':{
  'name': 'search_web',
  'description': 'Search the web',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the topic or subject to search on the web'},
}}}}

# Quickly test and see what a general web news search for Los Angeles yields
search_web(query="Los Angeles")

"snippet: A pair of vehicles, including a tank truck carrying bleach, overturned on the 105 Freeway in South Los Angeles early Saturday morning, prompting a major closure. According to the Los Angeles Fire Department, the collision was reported at 3:44 a.m. on the westbound lanes of the 105 near Vermont Avenue in the Vermont Vista neighborhood of LA., title: South Los Angeles 105 Freeway's westbound lanes blocked after truck overturns, spills bleach, link: https://www.cbsnews.com/losangeles/news/south-los-angeles-105-freeways-truck-spills-bleach/, date: 2026-05-09T14:00:00+00:00, source: CBS News, snippet: Authorities are searching for a possible missing person after an explosive fire ripped through the detached garage of a Los Angeles home early Saturday morning. The Los Angeles, title: 1 dead, 2 hurt after explosive Los Angeles garage fire, link: https://ktla.com/news/local-news/person-possibly-missing-after-explosive-fire-tears-through-garage-in-los-angeles/, date: 2026-05-09T15:10:

In [82]:
def search_ys(query: str) -> str:
  engine = DuckDuckGoSearchResults(backend="news")
  return engine.run(f"site:sports.yahoo.com {query}")

tool_search_ys = {'type':'function', 'function':{
  'name': 'search_ys',
  'description': 'Search for sports news',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the sport, sports team, or subject to search'},
}}}}

# Quickly test and see what a search for Los Angeles in the sports section of the news yields
search_ys(query="Los Angeles")

"snippet: Lakers trade idea sees Los Angeles cut ties with Luka Doncic's $54 million star teammate via trade originally appeared on The ..., title: Lakers trade idea sees Los Angeles cut ties with Luka Doncic's $54 million star teammate via trade, link: https://sports.yahoo.com/articles/lakers-idea-sees-los-angeles-011720360.html, date: 2026-05-08T22:24:13+00:00, source: Yahoo Sports, snippet: Dirk Nowitzki has weighed in on the growing officiating controversy surrounding the Los Angeles Lakers, and his take carried ..., title: Mavericks legend Dirk Nowitzki pokes fun at LA Lakers complaining about refs vs. OKC Thunder, link: https://sports.yahoo.com/articles/mavericks-legend-dirk-nowitzki-pokes-170000035.html, date: 2026-05-04T22:24:13+00:00, source: Yahoo Sports, snippet: The Los Angeles Angels are not having the season that they'd like to, sitting at the bottom of the American League standings., title: Los Angeles Angels' Recent Struggles Dissected On Roundtable's Podcast, link: htt

In [83]:
system_message = {
    "role": "system",
    "content": "You are a helpful assistant with access to tools for search the web for current news and events."
    }
user_message = {
    "role": "user",
    "content": "Tell me about the city of Denver." # YOU WILL CHANGE THIS QUESTION, SEE Q3 BELOW
}
messages = [system_message, user_message]

In [84]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant with access to tools for search the web for current news and events.'},
 {'role': 'user', 'content': 'Tell me about the city of Denver.'}]

In [85]:
response = ollama.chat(
  model="llama3.2:1b",
  tools=[tool_search_web, tool_search_ys],
  messages=messages
)
response

ChatResponse(model='llama3.2:1b', created_at='2026-05-09T22:24:22.0793759Z', done=True, done_reason='stop', total_duration=558512219, load_duration=193447296, prompt_eval_count=223, prompt_eval_duration=58892126, eval_count=28, eval_duration=241036395, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_web', arguments={'object': 'Denver', 'query': 'city of Denver'}))]), logprobs=None)

In [86]:
# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {'search_web':search_web, 'search_ys':search_ys}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 


TypeError: search_web() got an unexpected keyword argument 'object'

---

### Q3: The question above currently asks about Denver, but change the question to include a word or reference to sports. Does the agent use the correct tool based on your prompt/question? Be sure to also run the code cells above with your modified promp/question.

`<INSERT YOUR ANSWER HERE>`

---